# Inspect a parsed document

A helper for looking at what `ai_parse_document` produced for a PDF and how it came out as Markdown. It is **not** part of the job: open it in the workspace and run it for one document at a time.

1. **Overview**: pages, elements by type, and the document's history in `silver_log`.
2. **Page view**: the real PDF page with a coloured box around every parsed element (reading order and type), like the Databricks parse viewer.
3. **Elements table**: the page's elements with type, confidence and content.
4. **Parsed vs Markdown**: the text the parser found for the page next to the Markdown in `silver_pages`.
5. **Coverage**: which pages have content but no Markdown.
6. **All documents** (optional): the same coverage check for every document, and a summary of `silver_failures`, to find the ones to look at.

**How the boxes are drawn.** The parser gives each element's box in pixels of the page rendered at **200 DPI** (a US-letter page is 1700 x 2200). The notebook renders the page from the PDF in `bronze_pdfs` at 200 DPI and draws the boxes on it.

**Page numbers** are the real PDF page numbers (the parser's 0-based page id plus 1), the same as in `silver_pages` and in the `<!-- page N -->` markers.

In [ ]:
%pip install -q pymupdf
dbutils.library.restartPython()

In [ ]:
import base64, html, io, re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("doc", "BOEING_2022_10K")   # part of the PDF file name
dbutils.widgets.text("page", "5")                # PDF page to look at (1, 2, 3, ...)
dbutils.widgets.dropdown("run_all_docs", "no", ["no", "yes"])

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
doc = dbutils.widgets.get("doc")
page = int(dbutils.widgets.get("page"))
run_all_docs = dbutils.widgets.get("run_all_docs") == "yes"

bronze = f"{catalog}.{schema}.bronze_pdfs"
parsed_tbl = f"{catalog}.{schema}.silver_parsed"
pages_tbl = f"{catalog}.{schema}.silver_pages"
log_tbl = f"{catalog}.{schema}.silver_log"
failures_view = f"{catalog}.{schema}.silver_failures"
DPI = 200  # the parser's coordinates are in pixels of the page rendered at 200 DPI

## 1. Load the document and get an overview

Finds the document by a part of its file name (the `doc` widget). If the PDF is not in `silver_parsed` (its parse was incomplete or failed), the cell shows why, from `silver_log`.

In [ ]:
rows = spark.table(bronze).filter(f"path LIKE '%{doc}%'").select("path").collect()
assert len(rows) == 1, f"{len(rows)} PDFs match '{doc}': {[r.path for r in rows]}. Use a more specific name."
path = rows[0].path
print(path)

print("history in silver_log:")
display(spark.table(log_tbl).filter(f"path = '{path}'").orderBy("logged_at"))
assert spark.table(parsed_tbl).filter(f"path = '{path}'").count() == 1, "Not in silver_parsed: the parse was incomplete or failed. See the log above."

# One row per parsed element, in reading order.
els = spark.sql(f"""
SELECT e.pos AS pos,
       coalesce(e.value:bbox[0].page_id::INT, 0) + 1 AS page,
       e.value:type::STRING AS type,
       e.value:confidence::DOUBLE AS confidence,
       variant_get(e.value, '$.bbox[0].coord', 'ARRAY<INT>') AS coord,
       e.value:content::STRING AS content,
       e.value:description::STRING AS description
FROM (SELECT parsed FROM {parsed_tbl} WHERE path = '{path}') p,
     LATERAL variant_explode(p.parsed:document:elements) e
ORDER BY e.pos
""").toPandas()

info = spark.table(parsed_tbl).filter(f"path = '{path}'").select("pdf_pages", "pages").first()
converted = spark.table(pages_tbl).filter(f"path = '{path}'").count()
print(f"PDF pages: {info.pdf_pages}   pages with content: {info.pages}   elements: {len(els)}   pages converted: {converted}")
display(els.groupby("type").agg(elements=("pos", "count"), avg_chars=("content", lambda s: round(s.fillna("").str.len().mean()))).sort_values("elements", ascending=False).reset_index())

## 2. The page, with the parsed elements drawn on it

Renders the PDF page (from the bytes in `bronze_pdfs`) and draws a box around each element, coloured by type and numbered in reading order. Use it to check that titles, tables, headers and footers were found where they should be, and that nothing on the page was missed (text with no box).

In [ ]:
import fitz
from PIL import Image, ImageDraw

COLORS = {
    "title": "#d62728", "section_header": "#ff7f0e", "text": "#1f77b4", "table": "#2ca02c",
    "figure": "#9467bd", "caption": "#8c564b", "page_header": "#7f7f7f", "page_footer": "#7f7f7f",
    "page_number": "#bcbd22", "footnote": "#17becf", "list_item": "#e377c2",
}

pdf = fitz.open(stream=bytes(spark.table(bronze).filter(f"path = '{path}'").select("content").first()[0]), filetype="pdf")

def render_page(page_no, dpi=DPI):
    """PDF page `page_no` (1, 2, 3, ...) as an image with the element boxes drawn on it."""
    pix = pdf[page_no - 1].get_pixmap(matrix=fitz.Matrix(dpi / 72, dpi / 72))
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    draw = ImageDraw.Draw(img)
    for _, e in els[els["page"] == page_no].iterrows():
        if e["coord"] is None or len(e["coord"]) != 4:
            continue
        color = COLORS.get(e["type"], "#000000")
        x1, y1, x2, y2 = [int(v) for v in e["coord"]]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
        label = f"{e['pos']} {e['type']}"
        draw.rectangle([x1, max(y1 - 26, 0), x1 + 12 * len(label) + 8, max(y1, 26)], fill=color)
        draw.text((x1 + 4, max(y1 - 24, 0)), label, fill="white")
    return img

def show_image(img, width=900):
    img = img.copy(); img.thumbnail((width, width * 2))
    buf = io.BytesIO(); img.save(buf, format="PNG")
    displayHTML(f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}" style="max-width:100%">')

displayHTML(" ".join(f'<span style="background:{c};color:white;padding:2px 6px;margin:2px;border-radius:3px">{t}</span>' for t, c in COLORS.items()))
show_image(render_page(page))

## 3. The page's elements

In reading order. `pos` is the position in the whole document, `coord` is `[x1, y1, x2, y2]` in pixels at 200 DPI. Tables are HTML in `content`, and a figure may have a `description` made by the parser.

In [ ]:
page_els = els[els["page"] == page].copy()
page_els["content"] = page_els["content"].fillna("").str.slice(0, 200)
display(page_els[["pos", "type", "confidence", "coord", "content", "description"]])

## 4. Parsed text next to the Markdown

Left: the text the parser found on the page, element by element. Right: the Markdown in `silver_pages` for the same page. Look for dropped text, reworded text, a wrong heading level or a table that lost rows.

In [ ]:
row = spark.table(pages_tbl).filter(f"path = '{path}' AND page_no = {page}").first()
markdown = row.markdown if row else "(this page is not converted: see silver_log)"
parsed_text = "\n\n".join(f"[{r.type}] {r.content}" for r in els[els["page"] == page].itertuples() if r.content)
col = "width:49%;display:inline-block;vertical-align:top;white-space:pre-wrap;font-family:monospace;font-size:12px;padding:6px;border:1px solid #888;"
displayHTML(
    f'<div style="{col}"><b>Parsed (page {page})</b>\n\n{html.escape(parsed_text)}</div>'
    f'<div style="{col}"><b>Markdown (page {page})</b>\n\n{html.escape(markdown)}</div>'
)

## 5. Coverage: does every page with content have Markdown?

Compares the pages that have parsed elements with the pages in `silver_pages`. A page that is missing is either not converted yet, or it failed (look at `silver_log`). PDF pages with no elements at all (blank pages) have nothing to convert.

In [ ]:
content_pages = sorted(els["page"].unique())
done_pages = sorted(r.page_no for r in spark.table(pages_tbl).filter(f"path = '{path}'").select("page_no").collect())
missing = [p for p in content_pages if p not in done_pages]
print(f"pages with content: {len(content_pages)}   converted: {len(done_pages)}")
print(f"content but no Markdown: {missing or 'none'}")
print("OK" if not missing else "SOME PAGES ARE MISSING")

## 6. All documents (optional)

Set the `run_all_docs` widget to `yes` and rerun this cell. For every parsed document it shows the pages with content, the pages converted and the pages missing, worst first, and then a summary of what is open in `silver_failures`.

In [ ]:
if run_all_docs:
    coverage = spark.sql(f"""
    SELECT regexp_extract(p.path, '[^/]+$', 0) AS doc, p.pdf_pages, p.pages AS content_pages,
           coalesce(g.n, 0) AS converted, p.pages - coalesce(g.n, 0) AS missing
    FROM {parsed_tbl} p LEFT JOIN (SELECT path, count(*) AS n FROM {pages_tbl} GROUP BY path) g ON g.path = p.path
    ORDER BY missing DESC, doc
    """)
    display(coverage)
    print("documents with missing pages:", coverage.filter("missing > 0").count(), "of", coverage.count())
    display(spark.table(failures_view).groupBy("stage", "retryable").count())
else:
    print("Set the run_all_docs widget to yes to run this.")